# 09 — Revision analyses

Mechanistic and sensitivity analyses (paper Sections 3.10, 4.3–4.4, 4.7 and 4.9):
closed-form check and |z| regime (A5), matched variance decomposition (A1/A2),
reference / density / method sensitivity (A3/A4), survival re-analysis of new
representations, selection stability (A7), CGGA statistics (B3) and the MGMT check (B2).

The analysis code is in `scripts/`. This notebook only sets the paths and calls the scripts.
Run notebooks 01–08 first: these analyses read their outputs.

In [ ]:
import os, sys, importlib
from pathlib import Path
import pandas as pd

# Folder with the downloaded data and the outputs of notebooks 01-08
BASE = Path(os.environ.get("GBM_DATA_DIR", Path.home() / "Downloads"))
sys.path.insert(0, str(Path("scripts").resolve()))
import core

STAGE_C = BASE / "TCGA_GBM_STAGE_C_PERSONALIZED_REWIRING"
STAGE_D = BASE / "TCGA_GBM_STAGE_D_SURVIVAL"
GTEX  = STAGE_C / "STAGE_C_GTEX_334_REFERENCE_LOG_CPM_MTFIXED.csv"
TUMOR = BASE / "TCGA_GBM_STAGE_C_144_CALIBRATED" / "STAGE_C_144_CALIBRATED_RECOUNT3_SCALE.csv"
EDGES = STAGE_C / "STAGE_C_FINAL_FROZEN_EDGE_UNIVERSE.csv"
ENESH = STAGE_C / "STAGE_C_FINAL_REWIRING_MATRIX_144xANCHORS_20PCT.csv"
M3    = STAGE_D / "STAGE_D_M3_REWIRING_RAW.csv"
SPLIT = STAGE_D / "STAGE_D_LOCKED_SPLIT.csv"
SURV  = STAGE_D / "STAGE_D_SURVIVAL_MODELING_COHORT.csv"
QC    = next(BASE.rglob("STAGE_A_GTEX_CORTEX_FINAL_QC.csv"), BASE / "STAGE_A_GTEX_CORTEX_FINAL_QC.csv")
OUT   = BASE / "REVISION_RESULTS"
OUT.mkdir(exist_ok=True)
for f in (GTEX, TUMOR, EDGES, ENESH, M3, SPLIT, SURV, QC):
    print(("OK      " if f.exists() else "MISSING ") + str(f))

def run(module_name, args):
    """Run a script's main() with command-line arguments."""
    mod = importlib.import_module(module_name)
    importlib.reload(mod)
    sys.argv = [module_name] + [str(a) for a in args]
    mod.main()

# gene lists: 3,992 anchors and 3,669 modelling genes
(OUT / "anchors.txt").write_text("\n".join(core.read_matrix(str(ENESH)).index))
(OUT / "model_genes.txt").write_text("\n".join(core.read_matrix(str(M3)).index))

# GTEx region lists (BA9 and Brain - Cortex)
qc = pd.read_csv(QC, low_memory=False)
gtex_cols = set(core.read_matrix(str(GTEX)).columns)
ba9    = [s for s in qc.loc[qc["SMTSD"].str.contains("BA9"), "Recount sample ID"].astype(str) if s in gtex_cols]
cortex = [s for s in qc.loc[qc["SMTSD"].str.strip() == "Brain - Cortex", "Recount sample ID"].astype(str) if s in gtex_cols]
(OUT / "gtex_ba9.txt").write_text("\n".join(ba9))
(OUT / "gtex_cortex.txt").write_text("\n".join(cortex))
print("BA9:", len(ba9), " Cortex:", len(cortex), " (expected 217 and 117)")

## A5 — closed-form check (Eq. 4) and |z| regime

In [ ]:
run("a5_z_regime", ["--gtex", GTEX, "--tumour", TUMOR, "--edges", EDGES,
                    "--anchors", OUT / "anchors.txt", "--out", OUT / "a5"])

## A1/A2 — matched variance decomposition of eNESH

In [ ]:
run("a1_a2_variance_decomposition",
    ["--gtex", GTEX, "--tumour", TUMOR, "--edges", EDGES,
     "--anchors", OUT / "anchors.txt", "--dev", f"{SPLIT}:development",
     "--model-genes", OUT / "model_genes.txt", "--enesh", ENESH,
     "--dmax", 166, "--out", OUT / "a1"])

## A3/A4 — reference type × method at 20% density (1–2 h)

In [ ]:
run("a3_a4_network_sensitivity", [
    "--tumour", TUMOR, "--edges", EDGES, "--anchors", OUT / "anchors.txt",
    "--dev", f"{SPLIT}:development", "--model-genes", OUT / "model_genes.txt",
    "--decompose-genes", 800, "--skip-jaccard",
    "--reference", f"GTEX={GTEX}",
    "--reference", f"GTEX_BA9={GTEX}@{OUT / 'gtex_ba9.txt'}",
    "--reference", f"GTEX_CORTEX={GTEX}@{OUT / 'gtex_cortex.txt'}",
    "--gtex-halves", 2,
    "--reference", "TUMOUR_LOO",
    "--densities", "0.20", "--methods", "spcc", "ssn",
    "--save-enesh", "--out", OUT / "a3_pass1"])

## A3/A4 — densities 10%, 15% and FDR

In [ ]:
run("a3_a4_network_sensitivity", [
    "--tumour", TUMOR, "--edges", EDGES, "--anchors", OUT / "anchors.txt",
    "--dev", f"{SPLIT}:development", "--model-genes", OUT / "model_genes.txt",
    "--decompose-genes", 800, "--skip-jaccard",
    "--reference", f"GTEX={GTEX}", "--reference", "TUMOUR_LOO",
    "--densities", "0.10", "0.15", "fdr", "--methods", "spcc",
    "--out", OUT / "a3_pass2"])

## Feature matrices for TCGA and CGGA (eNESH binary / non-binary, R1, R2, R2b, expression)

In [ ]:
CGGA_HARM = next(BASE.rglob("CGGA_104_HARMONIZED_TO_GTEX.csv"))
CGGA_SURV = next(BASE.rglob("CGGA_SURVIVAL_COHORT.csv"))
run("make_feature_matrices", [
    "--gtex", GTEX, "--tumour", TUMOR, "--edges", EDGES, "--anchors", OUT / "anchors.txt",
    "--dev", f"{SPLIT}:development", "--model-genes", OUT / "model_genes.txt",
    "--external", CGGA_HARM, "--dmax", 166, "--out", OUT / "features"])

## Survival: same nested pipeline as the paper (about 1 h per representation)

In [ ]:
F = OUT / "features"
jobs = [
    ("M3_nonbinary",  F / "tcga_enesh_nonbinary.csv", F / "ext_enesh_nonbinary.csv"),
    ("M3_binary",     F / "tcga_enesh_binary.csv",    F / "ext_enesh_binary.csv"),
    ("M3_tumour_ref", OUT / "a3_pass1" / "enesh_TUMOUR_LOO_0.20_spcc.csv", None),
]
for name, feat, ext in jobs:
    args = ["--features", feat, "--survival", SURV, "--dev", f"{SPLIT}:development",
            "--holdout", f"{SPLIT}:locked", "--name", name, "--out", OUT / "survival"]
    if ext is not None:
        args += ["--external", ext, "--external-survival", CGGA_SURV]
    print("\n==========", name)
    run("nested_cv_survival", args)

## B3 — CGGA statistics: bootstrap CIs, paired differences, equivalence, minimum detectable difference

In [ ]:
PRED = {"M2": ("CGGA_104_M2_EXACT_FROZEN_PREDICTIONS.csv", "M2_frozen_risk"),
        "M3": ("CGGA_104_M3_AVAILABILITY_ADAPTED_PREDICTIONS.csv", "M3_availability_adapted_risk"),
        "R1": ("CGGA_104_R1_FROZEN_PREDICTIONS.csv", "R1_risk"),
        "R2": ("CGGA_104_R2_FROZEN_PREDICTIONS.csv", "R2_risk"),
        "R2b": ("CGGA_104_R2B_FROZEN_PREDICTIONS.csv", "R2B_risk")}
risks = None
for name, (fname, col) in PRED.items():
    d = pd.read_csv(next(BASE.rglob(fname)))
    idc = "CGGA_ID" if "CGGA_ID" in d.columns else d.columns[0]
    d = d[[idc, col]].rename(columns={idc: "sample", col: name})
    risks = d if risks is None else risks.merge(d, on="sample", how="outer")
d = pd.read_csv(next(BASE.rglob("CGGA_104_FINAL_M1_M4_M5_M6_PREDICTIONS.csv")))
idc = "CGGA_ID" if "CGGA_ID" in d.columns else d.columns[0]
num = [c for c in d.columns if c != idc and pd.api.types.is_numeric_dtype(d[c])]
pick = {}
for label, key in (("M1", "M1"), ("M4_M6", "M4")):
    hit = [c for c in num if key in c]
    if hit:
        pick[hit[0]] = label
d = d[[idc] + list(pick)].rename(columns={idc: "sample", **pick})
risks = risks.merge(d, on="sample", how="outer")
risks["sample"] = risks["sample"].astype(str)
(OUT / "b3").mkdir(exist_ok=True)
risks.to_csv(OUT / "b3" / "cgga_risks.csv", index=False)

dev_ids = set(core.read_list(f"{SPLIT}:development"))
s = pd.read_csv(SURV)
s[s["patient_id"].astype(str).isin(dev_ids)].to_csv(OUT / "b3" / "tcga_dev_surv.csv", index=False)

run("b3_transport_statistics", ["--risks", OUT / "b3" / "cgga_risks.csv", "--survival", CGGA_SURV,
    "--train-survival", OUT / "b3" / "tcga_dev_surv.csv", "--reference-model", "M3",
    "--margin", 0.05, "--out", OUT / "b3"])

## B2 — MGMT in CGGA: raw coding, log-rank in all patients, TMZ-treated and TMZ + RT

In [ ]:
raw = next(p for p in BASE.rglob("CGGA.mRNAseq_693_clinical.20200506.txt") if p.is_file())
cl = pd.read_csv(raw, sep="\t")
ids = set(pd.read_csv(OUT / "b3" / "cgga_risks.csv")["sample"].astype(str))
cl = cl[cl["CGGA_ID"].astype(str).isin(ids)].copy()
col = lambda key: next(c for c in cl.columns if key.lower() in c.lower())
mg = pd.DataFrame({"sample": cl["CGGA_ID"].astype(str),
                   "time": cl["OS"] / 30.4375,
                   "event": cl[col("Censor")],
                   "mgmt": cl[col("MGMT")].astype(str).str.strip(),
                   "tmz": cl[col("Chemo")], "rt": cl[col("Radio")]})
print(len(mg), "CGGA patients (expected 104)")
mg.to_csv(OUT / "b3" / "cgga_mgmt.csv", index=False)
run("b2_mgmt_check", ["--clinical", OUT / "b3" / "cgga_mgmt.csv", "--cohort", "CGGA",
    "--mgmt-column", "mgmt", "--methylated-values", "methylated",
    "--unmethylated-values", "un-methylated",
    "--tmz-column", "tmz", "--tmz-values", "1", "1.0",
    "--rt-column", "rt", "--rt-values", "1", "1.0", "--out", OUT / "b2"])

## A7 — selection stability (Jaccard and Kuncheva)

In [ ]:
L = STAGE_D / "REVISION_STEP2_DEVIATION_LADDER"
run("a7_selection_stability", [
    "--rep", f"M2={STAGE_D / 'STAGE_D_M2_EXPRESSION_RAW.csv'}",
    "--rep", f"M3={STAGE_D / 'STAGE_D_M3_REWIRING_RAW.csv'}",
    "--rep", f"R1={L / 'STAGE_D_R1_ABS_DEVIATION_RAW.csv'}",
    "--rep", f"R2={L / 'STAGE_D_R2_NEIGHBOUR_ABS_DEVIATION_RAW.csv'}",
    "--rep", f"R2b={L / 'STAGE_D_R2B_GENE_X_NEIGHBOUR_DEVIATION_RAW.csv'}",
    "--survival", SURV, "--dev", f"{SPLIT}:development", "--all", SPLIT,
    "--out", OUT / "a7"])